# L06 Missed appointments (synthetic data)
AI-25 AI in Healthcare. All data in this notebook is **synthetic**: it is generated by the code below and describes no real patients. Do not upload real patient data to Colab.

Run each cell in order (Runtime > Run all).

## Step 1: create the synthetic appointment dataset (2,000 rows)

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(25)
n = 2000
df = pd.DataFrame({
    "distance_km": rng.gamma(2.0, 6.0, n).round(1),
    "lead_days": rng.integers(0, 43, n),
    "reminder_sent": rng.integers(0, 2, n),
    "age_group": rng.choice(["18-39", "40-64", "65+"], n),
    "previous_missed": rng.poisson(0.5, n),
})
logit = (-2.0 + 0.05*df.distance_km + 0.03*df.lead_days - 0.8*df.reminder_sent
         + 0.5*df.previous_missed)
df["missed"] = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
df.to_csv("synthetic_appointments.csv", index=False)


## Step 2: missed-appointment rate by factor, and a simple logistic regression

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
df = pd.read_csv("synthetic_appointments.csv")
print(df.shape, "missed rate:", round(df.missed.mean(), 3))
df["distance_band"] = pd.cut(df.distance_km, [0, 5, 15, 30, 200],
                             labels=["0-5", "5-15", "15-30", "30+"])
for col in ["distance_band", "reminder_sent", "age_group"]:
    print(df.groupby(col, observed=True).missed.agg(["mean", "count"]).round(3), "\n")
X = df[["distance_km", "lead_days", "reminder_sent", "previous_missed"]]
model = LogisticRegression(max_iter=1000).fit(X, df.missed)
print(pd.Series(model.coef_[0], index=X.columns).round(3))


## Step 3: your notes
Which factors relate to missed appointments? Propose two fair actions (for example reminders or transport support). Relationships here are associations in synthetic data, not causes.